In [36]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import mlflow
import mlflow.pyfunc
from datetime import datetime
import random
import dagshub   
from mlflow.tracking import MlflowClient 
import json 
import pandas as pd
import numpy as np
import random
from sklearn.metrics import ndcg_score
from scipy.stats import entropy
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

In [2]:
dagshub.init(repo_owner='bteinstein', repo_name='demand_engine', mlflow=True)

Accessing as bteinstein

Initialized MLflow to track repo "bteinstein/demand_engine"

Repository bteinstein/demand_engine initialized!

### List Top Models

In [3]:
from mlflow.tracking import MlflowClient

# Initialize the MLflow client
client = MlflowClient()

# Search for all registered models
registered_models = client.search_registered_models()

# Iterate over each registered model
for rm in registered_models:
    print(f"Model Name: {rm.name}")

    # Search for all versions of the model
    model_versions = client.search_model_versions(f"name='{rm.name}'")

    # Iterate over each version
    for version in model_versions:
        print(f"  Version: {version.version}")
        print(f"  Stage: {version.current_stage}")
        print(f"  Run ID: {version.run_id}")
        print(f"  Creation Timestamp: {version.creation_timestamp}")
        print(f"  Last Updated Timestamp: {version.last_updated_timestamp}")
        # print(f"  Source: {version.source}")
        print("---")


Model Name: BalancedRandomForest_20250527_171136
  Version: 1
  Stage: None
  Run ID: a7c7b78e04f94db08e3cfc52fc4230b4
  Creation Timestamp: 1748422984751
  Last Updated Timestamp: 1748422984751
---
Model Name: EasyEnsemble
Model Name: EasyEnsemble_20250527_171207
  Version: 2
  Stage: None
  Run ID: 54eac8269b204a06ac8670b6e0eeaf62
  Creation Timestamp: 1748422983968
  Last Updated Timestamp: 1748422983968
---
Model Name: EasyEnsemble-C1
Model Name: EasyEnsembleModel
Model Name: ModelEasyEnsemble
  Version: 1
  Stage: None
  Run ID: 54eac8269b204a06ac8670b6e0eeaf62
  Creation Timestamp: 1748418384711
  Last Updated Timestamp: 1748418384711
---
Model Name: RF_XGBoost_Ensemble_20250527_171121
  Version: 1
  Stage: None
  Run ID: 522fcb66a19a447dbdbe3545da8426bb
  Creation Timestamp: 1748422985532
  Last Updated Timestamp: 1748422985532
---


### Set Up Defaults

In [4]:
# Constants
EXPERIMENT_NAME = "sku_purchase_likelihood_experiment"
NUM_WEEKS = 10  # Number of weeks in training data
N_SPLITS = 3  # Number of temporal cross-validation splits 
MAX_DAYS = 365  # For imputing DaysSinceLastPurchase_SKU
MODEL_NAME="RF_XGBoost_Ensemble_20250527_171121"
MODEL_VERSION=1

### Inference Functions

In [5]:
# Set up MLflow tracking
def setup_mlflow():
    dagshub.init(repo_owner='bteinstein', repo_name='demand_engine', mlflow=True)
    mlflow.set_experiment(EXPERIMENT_NAME)

In [6]:
# Preprocess inference data
def preprocess_inference_data_(inference_data, numerical_features, label_encoders=None):
    # Impute missing values
    fill_values = {col: 0 for col in numerical_features}
    for col in numerical_features:
        if col.startswith('DaysSinceLastPurchase'):
            fill_values[col] = MAX_DAYS + 1
    inference_data = inference_data.fillna(fill_values)
    
    # Label encode categorical features
    if label_encoders is None:
        label_encoders = {}
        for col in ['CustomerID', 'SKUID']:
            le = LabelEncoder()
            inference_data[col] = le.fit_transform(inference_data[col])
            label_encoders[col] = le
    else:
        for col in ['CustomerID', 'SKUID']:
            le = label_encoders.get(col)
            if le:
                # Handle unseen labels by mapping to a default value (e.g., max label + 1)
                inference_data[col] = inference_data[col].map(lambda x: x if x in le.classes_ else 'Unknown')
                inference_data.loc[inference_data[col] == 'Unknown', col] = len(le.classes_)
                inference_data[col] = le.transform(inference_data[col])
    
    return inference_data, label_encoders


In [7]:
# New 
def preprocess_inference_data(inference_data, numerical_features, label_encoders=None):
    # Backup original values
    inference_data['CustomerID_original'] = inference_data['CustomerID']
    inference_data['SKUID_original'] = inference_data['SKUID']
    
    # Impute missing values
    fill_values = {col: 0 for col in numerical_features}
    for col in numerical_features:
        if col.startswith('DaysSinceLastPurchase'):
            fill_values[col] = MAX_DAYS + 1
    inference_data = inference_data.fillna(fill_values)

    # Label encode categorical features
    if label_encoders is None:
        label_encoders = {}
        for col in ['CustomerID', 'SKUID']:
            le = LabelEncoder()
            inference_data[col] = le.fit_transform(inference_data[col])
            label_encoders[col] = le
    else:
        for col in ['CustomerID', 'SKUID']:
            le = label_encoders.get(col)
            if le:
                inference_data[col] = inference_data[col].map(lambda x: x if x in le.classes_ else 'Unknown')
                inference_data.loc[inference_data[col] == 'Unknown', col] = len(le.classes_)
                inference_data[col] = le.transform(inference_data[col])
    
    return inference_data, label_encoders


In [8]:
model_name = MODEL_NAME
model_version = MODEL_VERSION
print(f"Loading model {model_name} version {model_version}...")
# model_version = client.get_model_version(model_name, model_version)

Loading model RF_XGBoost_Ensemble_20250527_171121 version 1...


In [9]:
# Load a model in its native flavor from MLflow
def load_native_model(model_name, model_version=None):
    # Initialize the MLflow client
    client = MlflowClient()

    # Retrieve the run
    model_version = client.get_model_version(model_name, model_version)
    run_id = model_version.run_id 
    run = client.get_run(run_id)    
    print(f"Resolved model {model_name} version {model_version}, run_id: {run_id}")

    # Extract the log-model history
    log_model_history = run.data.tags['mlflow.log-model.history']

    # Parse the JSON string
    model_history = json.loads(log_model_history)

    # Extract model details
    model_info = model_history[0]
    artifact_path = model_info['artifact_path']
    model_uuid = model_info['model_uuid']
    
    # Determine the model flavor
    flavors = model_info['flavors'] 
    model_flavor = None
    if 'sklearn' in flavors:
        model_flavor = 'sklearn'
    elif 'xgboost' in flavors:
        model_flavor = 'xgboost'
    elif 'lightgbm' in flavors:
        model_flavor = 'lightgbm'
    elif 'catboost' in flavors:
        model_flavor = 'catboost'

    print(f"Artifact Path: {artifact_path}")
    print(f"Model UUID: {model_uuid}")
    print(f"Model Flavor: {model_flavor}")  
    
    if model_flavor is None:
        raise ValueError("Unsupported or unknown model flavor.")

    # Load the model   
    # Construct the model URI
    model_uri = f"runs:/{run_id}/{artifact_path}"

    # Load the model in its native flavor
    try:
        if model_flavor == 'sklearn':
            model = mlflow.sklearn.load_model(model_uri)
        elif model_flavor == 'xgboost':
            model = mlflow.xgboost.load_model(model_uri)
        elif model_flavor == 'lightgbm':
            model = mlflow.lightgbm.load_model(model_uri)
        elif model_flavor == 'catboost':
            model = mlflow.catboost.load_model(model_uri)
        else:
            raise ValueError(f"Unsupported model flavor: {model_flavor}")
        
        return model
    except Exception as e:
        raise ValueError(f"Failed to load model in native flavor {model_flavor}: {str(e)}")


In [10]:
# Run predictions
def run_predictions_(model, inference_data, numerical_features, 
                    features, is_logistic=False, scaler=None,
                    output_path='./output/predictions.csv'):
    # Scale numerical features if LogisticRegression
    if is_logistic:
        if scaler is None:
            scaler = StandardScaler()    
        inference_data[numerical_features] = scaler.transform(inference_data[numerical_features])
           
    # Predict
    # Predict probabilities if the model supports it
    if hasattr(model, 'predict_proba'):        
        predictions = model.predict_proba(inference_data[features])[:, 1]
        inference_data['purchase_likelihood'] = predictions
    else:
        print("The model does not support predict_proba.")
        
    
    # Save predictions
    output_cols = ['Week', 'CustomerID', 'SKUID', 'purchase_likelihood']
    inference_data[output_cols].to_csv(output_path, index=False)
    print(f"Predictions saved to '{output_path}")
    return inference_data


In [11]:
def run_predictions(model, inference_data, numerical_features, 
                    features, is_logistic=False, scaler=None,
                    output_path='./output/predictions.csv'):
    # Scale numerical features if LogisticRegression
    if is_logistic:
        if scaler is None:
            scaler = StandardScaler()    
        inference_data[numerical_features] = scaler.transform(inference_data[numerical_features])
    
    # Predict
    if hasattr(model, 'predict_proba'):
        predictions = model.predict_proba(inference_data[features])[:, 1]
        inference_data['purchase_likelihood'] = predictions
    else:
        print("The model does not support predict_proba.")
    
    # Save predictions with original IDs
    predictions_df = inference_data.copy()
    predictions_df['CustomerID'] = inference_data['CustomerID_original']
    predictions_df['SKUID'] = inference_data['SKUID_original'] 
    
    output_cols = ['Week', 'CustomerID_original', 'SKUID_original', 'purchase_likelihood']
    inference_data[output_cols].to_csv(output_path, index=False)
    print(f"Predictions saved to '{output_path}'")
    return predictions_df


In [ ]:

# Evaluate predictions for random customers with category validation
def evaluate_customer_predictions(training_data, predictions, num_customers=3, top_k=5):
    skus = pd.read_csv('./data/skus.csv')
    training_data = training_data.merge(skus[['SKUID', 'Category']], on='SKUID', how='left')
    predictions = predictions.merge(skus[['SKUID', 'Category']], on='SKUID', how='left')
    
    # Filter historical purchases
    historical_purchases = training_data[training_data['purchase_likelihood'] == 1][['CustomerID', 'SKUID', 'Category', 'Week']]
    
    customers = training_data['CustomerID'].unique()
    random.seed(42)
    selected_customers = random.sample(list(customers), num_customers)
    
    evaluation_results = []
    
    for customer in selected_customers:
        customer_history = historical_purchases[historical_purchases['CustomerID'] == customer]
        purchased_skus = customer_history['SKUID'].unique()
        purchased_categories = customer_history['Category'].unique()
        purchase_counts = customer_history.groupby('SKUID').size().sort_values(ascending=False)
        category_counts = customer_history.groupby('Category').size().sort_values(ascending=False)
        recent_purchases = customer_history.sort_values('Week', ascending=False).head(5)
        
        customer_predictions = predictions[predictions['CustomerID'] == customer][['SKUID', 'Category', 'purchase_likelihood']]
        top_predicted_skus = customer_predictions.nlargest(top_k, 'purchase_likelihood')
        
        predicted_skus = top_predicted_skus['SKUID'].values
        sku_overlap = len(set(predicted_skus) & set(purchased_skus)) / top_k
        freq_scores = [purchase_counts.get(sku, 0) for sku in predicted_skus]
        mean_freq = np.mean(freq_scores) if freq_scores else 0
        predicted_categories = top_predicted_skus['Category'].values
        category_overlap = len(set(predicted_categories) & set(purchased_categories)) / top_k
        
        validation_notes = []
        for _, row in top_predicted_skus.iterrows():
            sku = row['SKUID']
            category = row['Category']
            if sku in purchased_skus:
                last_week = customer_history[customer_history['SKUID'] == sku]['Week'].max()
                freq = purchase_counts.get(sku, 0)
                validation_notes.append(f"SKU {sku} (Category {category}): Purchased {freq} times, last in Week {last_week}")
            else:
                if category in purchased_categories:
                    cat_freq = category_counts.get(category, 0)
                    validation_notes.append(f"SKU {sku} (Category {category}): Not purchased, but category purchased {cat_freq} times")
                else:
                    validation_notes.append(f"SKU {sku} (Category {category}): Not purchased, category not historically bought")
        
        evaluation_results.append({
            'CustomerID': customer,
            'Top_Predicted_SKUs': top_predicted_skus.to_dict('records'),
            'Historical_Purchases': purchased_skus.tolist(),
            'Historical_Categories': purchased_categories.tolist(),
            'SKU_Overlap_Proportion': sku_overlap,
            'Category_Overlap_Proportion': category_overlap,
            'Mean_Purchase_Frequency': mean_freq,
            'Validation_Notes': validation_notes,
            'Recent_Purchases': recent_purchases[['SKUID', 'Category', 'Week']].to_dict('records')
        })
    
    for result in evaluation_results:
        # print line breaks to separate evaluations
        print("\n" + "="*50 + "\n")
        print(f"\nEvaluation for Customer {result['CustomerID']}:")
        print("Top 5 Predicted SKUs:")
        for pred in result['Top_Predicted_SKUs']:
            print(f"  - SKU {pred['SKUID']} (Category {pred['Category']}): Likelihood = {pred['purchase_likelihood']:.4f}")
        print(f"SKU Overlap with Historical Purchases: {result['SKU_Overlap_Proportion']:.2f}")
        print(f"Category Overlap with Historical Purchases: {result['Category_Overlap_Proportion']:.2f}")
        print(f"Mean Purchase Frequency of Predicted SKUs: {result['Mean_Purchase_Frequency']:.2f}")
        print("Validation Notes:")
        for note in result['Validation_Notes']:
            print(f"  - {note}")
        print("Recent Purchases:")
        for purchase in result['Recent_Purchases']:
            print(f"  - SKU {purchase['SKUID']} (Category {purchase['Category']}) in Week {purchase['Week']}")
    
    return evaluation_results


In [13]:
def main(inference_data_path='./input/inference_data_multi_windows.csv',
         training_data_path='./input/training_data_multi_windows.csv',
         MODEL_NAME=MODEL_NAME,
         MODEL_VERSION=MODEL_VERSION,
         label_encoders_path='./model_artifacts/label_encoders.pkl',
         scaler_path='./model_artifacts/scaler.pkl',
         prediction_output_path='./output/predictions.csv'):
    # Setup MLflow
    setup_mlflow()
    
    # Load Model  
    model = load_native_model(model_name = MODEL_NAME, model_version=MODEL_VERSION)
    # model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
    print(f"Loaded model: {MODEL_NAME} version")
    
    # Step 3: Load and preprocess inference data
    inference_data = pd.read_csv(inference_data_path)
    training_data = pd.read_csv(training_data_path)
    numerical_features = [col for col in inference_data.columns if col not in ['Week', 'CustomerID', 'SKUID']]
    features = numerical_features + ['CustomerID', 'SKUID', 'Week']
    
    # Load or create label encoders
    try:
        import pickle
        with open(label_encoders_path, 'rb') as f:
            label_encoders = pickle.load(f)
    except FileNotFoundError:
        label_encoders = None
        
    # Load or create label scaler
    try:
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
    except FileNotFoundError:
        scaler = None 
    
    inference_data, label_encoders = preprocess_inference_data(inference_data, numerical_features, label_encoders)
    
    # # Save label encoders for future use
    # with open(label_encoders_path, 'wb') as f:
    #     pickle.dump(label_encoders, f)
    
    # Step 4: Run predictions
    is_logistic = False  # Set to True if using LogisticRegression
    predictions = run_predictions(model, inference_data, numerical_features, features, is_logistic, scaler, prediction_output_path) 
    
    # Step 5: Evaluate predictions for 3 random customers
    evaluation_results = evaluate_customer_predictions(training_data, predictions)
    
    # Log evaluation results to MLflow
    with mlflow.start_run(run_name=f"Evaluation_{MODEL_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
        for i, result in enumerate(evaluation_results):
            mlflow.log_metric(f"Customer_{result['CustomerID']}_Overlap", result['Overlap_Proportion'])
            mlflow.log_metric(f"Customer_{result['CustomerID']}_MeanFreq", result['Mean_Purchase_Frequency'])
        evaluation_df = pd.DataFrame(evaluation_results)
        evaluation_df.to_csv('./evaluation/customer_evaluation.csv', index=False)
        mlflow.log_artifact('./evaluation/customer_evaluation.csv')


## Run Inference

In [ ]:

if __name__ == "__main__":
    main()

--------------------------

## MANUAL RUN

In [ ]:
inference_data_path='./input/inference_data_multi_windows.csv'
training_data_path='./input/training_data_multi_windows.csv'
MODEL_NAME="ModelEasyEnsemble"
MODEL_VERSION = 1
MODEL_ALIAS='champion'
label_encoders_path='./model_artifacts/label_encoders.pkl'
scaler_path='./model_artifacts/scaler.pkl'
prediction_output_path='./output/predictions.csv'
inference_data = pd.read_csv(inference_data_path)
training_data = pd.read_csv(training_data_path)
numerical_features = [col for col in inference_data.columns if col not in ['Week', 'CustomerID', 'SKUID']]
features = numerical_features + ['CustomerID', 'SKUID', 'Week']

# Load or create label encoders
try:
    import pickle
    with open(label_encoders_path, 'rb') as f:
        label_encoders = pickle.load(f)
except FileNotFoundError:
    label_encoders = None
    
# Load or create label scaler
try:
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)
except FileNotFoundError:
    scaler = None 

model = load_native_model(model_name = MODEL_NAME, model_version=MODEL_VERSION)
inference_data, label_encoders = preprocess_inference_data(inference_data, numerical_features, label_encoders)

In [18]:
# Step 4: Run predictions
is_logistic = False  # Set to True if using LogisticRegression
predictions = run_predictions(model, inference_data, numerical_features, features, is_logistic, scaler, prediction_output_path) 


Predictions saved to './output/predictions.csv'


In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.metrics import ndcg_score
from scipy.stats import entropy
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

def evaluate_customer_predictions_enhanced(training_data, predictions, num_customers=3, 
                                           k_values=[5], holdout_weeks=None, seed=42):
    """
    Enhanced evaluation of SKU purchase likelihood predictions for selected customers.
    
    Args:
        training_data (pd.DataFrame): Training data with CustomerID, SKUID, Week, purchase_likelihood.
        predictions (pd.DataFrame): Predictions with CustomerID, SKUID, purchase_likelihood.
        num_customers (int): Number of customers to evaluate (default: 3).
        k_values (list): List of top-K values to evaluate (default: [5]).
        holdout_weeks (list, optional): Weeks for holdout evaluation (e.g., [9, 10]).
        seed (int): Random seed for reproducibility (default: 42).
    
    Returns:
        tuple: (evaluation_results, agg_metrics) where evaluation_results is a list of per-customer metrics,
               and agg_metrics is a dict of aggregated metrics.
    """
    # Load SKU categories
    try:
        skus = pd.read_csv('./data/skus.csv')
    except FileNotFoundError:
        raise FileNotFoundError("skus.csv not found in ./data/")

    # Merge categories
    training_data = training_data.merge(skus[['SKUID', 'Category']], on='SKUID', how='left')
    predictions = predictions.merge(skus[['SKUID', 'Category']], on='SKUID', how='left')
    
    # Handle missing categories
    training_data['Category'].fillna('Unknown', inplace=True)
    predictions['Category'].fillna('Unknown', inplace=True)
    
    # Filter historical purchases
    if holdout_weeks:
        historical_purchases = training_data[training_data['Week'].isin(holdout_weeks) & 
                                            (training_data['purchase_likelihood'] == 1)][['CustomerID', 'SKUID', 'Category', 'Week']]
    else:
        historical_purchases = training_data[training_data['purchase_likelihood'] == 1][['CustomerID', 'SKUID', 'Category', 'Week']]
    
    # Select random customers
    customers = training_data['CustomerID'].unique()
    random.seed(seed)
    selected_customers = random.sample(list(customers), min(num_customers, len(customers)))
    
    evaluation_results = []
    
    for customer in selected_customers:
        customer_history = historical_purchases[historical_purchases['CustomerID'] == customer]
        if customer_history.empty:
            print(f"Skipping Customer {customer}: No historical purchases")
            evaluation_results.append({
                'CustomerID': customer,
                'Top_K': k_values[0],
                'Top_Predicted_SKUs': [],
                'Historical_Purchases': [],
                'Historical_Categories': [],
                'SKU_Precision': 0,
                'SKU_Recall': 0,
                'SKU_F1_Score': 0,
                'Category_Overlap_Proportion': 0,
                'Mean_Purchase_Frequency': 0,
                'NDCG': 0,
                'MRR': 0,
                'Mean_Recency_Weeks': 0,
                'Recent_Matches': 0,
                'Unique_Categories': 0,
                'Category_Entropy': 0,
                'Validation_Notes': ["No historical purchases"],
                'Recent_Purchases': []
            })
            continue
        
        purchased_skus = customer_history['SKUID'].unique()
        purchased_categories = customer_history['Category'].unique()
        purchase_counts = customer_history.groupby('SKUID').size().sort_values(ascending=False)
        category_counts = customer_history.groupby('Category').size().sort_values(ascending=False)
        recent_purchases = customer_history.sort_values('Week', ascending=False).head(5)
        max_week = customer_history['Week'].max()
        
        customer_predictions = predictions[predictions['CustomerID'] == customer][['SKUID', 'Category', 'purchase_likelihood']]
        if customer_predictions.empty:
            print(f"Skipping Customer {customer}: No predictions available")
            continue
        
        for k in k_values:
            top_predicted_skus = customer_predictions.nlargest(k, 'purchase_likelihood')
            predicted_skus = top_predicted_skus['SKUID'].values
            predicted_categories = top_predicted_skus['Category'].values
            
            # SKU and Category Metrics
            sku_precision = len(set(predicted_skus) & set(purchased_skus)) / k
            sku_recall = len(set(predicted_skus) & set(purchased_skus)) / max(1, len(purchased_skus))
            sku_f1 = 2 * (sku_precision * sku_recall) / (sku_precision + sku_recall) if (sku_precision + sku_recall) > 0 else 0
            category_overlap = len(set(predicted_categories) & set(purchased_categories)) / k
            
            # Purchase Frequency
            freq_scores = [purchase_counts.get(sku, 0) for sku in predicted_skus]
            mean_freq = np.mean(freq_scores) if freq_scores else 0
            
            # Ranking Metrics
            relevance = [1 if sku in purchased_skus else 0 for sku in predicted_skus]
            ndcg = ndcg_score([sorted(relevance, reverse=True)], [relevance], k=k) if sum(relevance) > 0 else 0
            mrr = next((1 / (i + 1) for i, sku in enumerate(predicted_skus) if sku in purchased_skus), 0)
            
            # Temporal Analysis
            recency_scores = [max_week - customer_history[customer_history['SKUID'] == sku]['Week'].max()
                             if sku in purchased_skus else np.inf for sku in predicted_skus]
            mean_recency = np.mean([r for r in recency_scores if r != np.inf]) if any(r != np.inf for r in recency_scores) else 0
            recent_matches = sum(1 for r in recency_scores if r <= 2)
            
            # Diversity Metrics
            unique_categories = len(set(predicted_categories))
            category_dist = pd.Series(predicted_categories).value_counts(normalize=True)
            category_entropy = entropy(category_dist) if not category_dist.empty else 0
            
            # Validation Notes
            validation_notes = []
            for _, row in top_predicted_skus.iterrows():
                sku = row['SKUID']
                category = row['Category']
                if sku in purchased_skus:
                    last_week = customer_history[customer_history['SKUID'] == sku]['Week'].max()
                    freq = purchase_counts.get(sku, 0)
                    validation_notes.append(f"SKU {sku} (Category {category}): Purchased {freq} times, last in Week {last_week}")
                else:
                    if category in purchased_categories:
                        cat_freq = category_counts.get(category, 0)
                        validation_notes.append(f"SKU {sku} (Category {category}): Not purchased, but category purchased {cat_freq} times")
                    else:
                        validation_notes.append(f"SKU {sku} (Category {category}): Not purchased, category not historically bought")
            
            evaluation_results.append({
                'CustomerID': customer,
                'Top_K': k,
                'Top_Predicted_SKUs': top_predicted_skus.to_dict('records'),
                'Historical_Purchases': purchased_skus.tolist(),
                'Historical_Categories': purchased_categories.tolist(),
                'SKU_Precision': sku_precision,
                'SKU_Recall': sku_recall,
                'SKU_F1_Score': sku_f1,
                'Category_Overlap_Proportion': category_overlap,
                'Mean_Purchase_Frequency': mean_freq,
                'NDCG': ndcg,
                'MRR': mrr,
                'Mean_Recency_Weeks': mean_recency,
                'Recent_Matches': recent_matches,
                'Unique_Categories': unique_categories,
                'Category_Entropy': category_entropy,
                'Validation_Notes': validation_notes,
                'Recent_Purchases': recent_purchases[['SKUID', 'Category', 'Week']].to_dict('records')
            })
    
    # Probability Calibration
    validation_data = training_data[training_data['CustomerID'].isin(selected_customers)]
    merged = validation_data.merge(predictions, on=['CustomerID', 'SKUID'], suffixes=('_true', '_pred'))
    try:
        prob_true, prob_pred = calibration_curve(merged['purchase_likelihood_true'], 
                                                merged['purchase_likelihood_pred'], n_bins=10)
        ece = np.mean(np.abs(prob_true - prob_pred))
    except:
        ece = np.nan
        print("Warning: Calibration curve failed due to insufficient data")
    
    # Aggregate Metrics
    agg_metrics = {
        'Mean_SKU_Precision': np.mean([r['SKU_Precision'] for r in evaluation_results]),
        'Mean_SKU_Recall': np.mean([r['SKU_Recall'] for r in evaluation_results]),
        'Mean_SKU_F1_Score': np.mean([r['SKU_F1_Score'] for r in evaluation_results]),
        'Mean_Category_Overlap': np.mean([r['Category_Overlap_Proportion'] for r in evaluation_results]),
        'Mean_Purchase_Frequency': np.mean([r['Mean_Purchase_Frequency'] for r in evaluation_results]),
        'Mean_NDCG': np.mean([r['NDCG'] for r in evaluation_results]),
        'Mean_MRR': np.mean([r['MRR'] for r in evaluation_results]),
        'Mean_Recency_Weeks': np.mean([r['Mean_Recency_Weeks'] for r in evaluation_results if r['Mean_Recency_Weeks'] > 0]) or 0,
        'Expected_Calibration_Error': ece
    }
    
    # Actionable Insights
    insights = []
    if agg_metrics['Mean_SKU_Precision'] < 0.1:
        insights.append("Low SKU precision suggests model fails to predict historical purchases. "
                        "Check feature engineering (e.g., purchase frequency, recency) or retrain with recent data.")
    if agg_metrics['Mean_Category_Overlap'] > agg_metrics['Mean_SKU_Precision']:
        insights.append("Higher category overlap indicates model captures general preferences but not specific SKUs. "
                        "Consider SKU-level features.")
    if agg_metrics['Mean_Recency_Weeks'] > 4:
        insights.append("High mean recency suggests predictions favor older purchases. "
                        "Incorporate temporal features to prioritize recent behavior.")
    if agg_metrics['Expected_Calibration_Error'] > 0.1 and not np.isnan(agg_metrics['Expected_Calibration_Error']):
        insights.append("High calibration error indicates miscalibrated probabilities. "
                        "Consider probability calibration (e.g., Platt scaling).")
    
    # Visualizations
    plt.figure(figsize=(10, 6))
    customers = [r['CustomerID'] for r in evaluation_results]
    plt.bar(customers, [r['SKU_Precision'] for r in evaluation_results], label='SKU Precision', alpha=0.5)
    plt.bar(customers, [r['Category_Overlap_Proportion'] for r in evaluation_results], label='Category Overlap', alpha=0.5)
    plt.xlabel('CustomerID')
    plt.ylabel('Metric Value')
    plt.title('SKU and Category Overlap per Customer')
    plt.legend()
    plt.savefig("overlap_plot.png")
    plt.close()
    
    # Print Results
    for result in evaluation_results:
        print("\n" + "="*50 + "\n")
        print(f"\nEvaluation for Customer {result['CustomerID']} (Top {result['Top_K']}):")
        print("Top Predicted SKUs:")
        for pred in result['Top_Predicted_SKUs']:
            print(f"  - SKU {pred['SKUID']} (Category {pred['Category']}): Likelihood = {pred['purchase_likelihood']:.4f}")
        print(f"SKU Precision: {result['SKU_Precision']:.2f}")
        print(f"SKU Recall: {result['SKU_Recall']:.2f}")
        print(f"SKU F1-Score: {result['SKU_F1_Score']:.2f}")
        print(f"Category Overlap: {result['Category_Overlap_Proportion']:.2f}")
        print(f"Mean Purchase Frequency: {result['Mean_Purchase_Frequency']:.2f}")
        print(f"NDCG: {result['NDCG']:.2f}")
        print(f"MRR: {result['MRR']:.2f}")
        print(f"Mean Recency (Weeks): {result['Mean_Recency_Weeks']:.2f}")
        print(f"Recent Matches (≤2 Weeks): {result['Recent_Matches']}")
        print(f"Unique Categories: {result['Unique_Categories']}")
        print(f"Category Entropy: {result['Category_Entropy']:.2f}")
        print("Validation Notes:")
        for note in result['Validation_Notes']:
            print(f"  - {note}")
        print("Recent Purchases:")
        for purchase in result['Recent_Purchases']:
            print(f"  - SKU {purchase['SKUID']} (Category {purchase['Category']}) in Week {purchase['Week']}")
    
    print("\nAggregate Metrics:")
    for key, value in agg_metrics.items():
        print(f"{key}: {value:.2f}")
    
    print("\nActionable Insights:")
    for insight in insights:
        print(f"- {insight}")
    
    print(f"\nVisualization saved to overlap_plot.png")
    
    return evaluation_results, agg_metrics

In [ ]:
# Step 5: Evaluate predictions for 3 random customers
# evaluation_results = evaluate_customer_predictions(training_data, predictions)
evaluation_results = evaluate_customer_predictions_enhanced(training_data, predictions, num_customers=2, k_values=[5], holdout_weeks=None, seed=42)

In [24]:
print(training_data.columns.tolist())
# print(inference_data.columns.tolist())
print(predictions.columns.tolist())

['Week', 'CustomerID', 'SKUID', 'purchase_likelihood', 'DaysSinceLastPurchase_SKU', 'TotalPurchases_SKU', 'AvgOrderValue_SKU_by_Customer', 'Customer_Recency', 'Customer_Frequency', 'Customer_Monetary', 'Customer_TotalUniqueSKUsPurchased_Overall', 'SKU_TotalSales_CustomerState_Overall', 'Customer_RollingAvgOrderValue_4Weeks', 'Customer_RollingUniqueSKUsPurchased_4Weeks', 'Customer_RollingPurchaseCount_4Weeks', 'SKU_RollingTotalSales_4Weeks', 'SKU_RollingPurchaseCount_4Weeks', 'Category_RollingTotalSales_4Weeks', 'Customer_RollingAvgOrderValue_8Weeks', 'Customer_RollingUniqueSKUsPurchased_8Weeks', 'Customer_RollingPurchaseCount_8Weeks', 'SKU_RollingTotalSales_8Weeks', 'SKU_RollingPurchaseCount_8Weeks', 'Category_RollingTotalSales_8Weeks', 'Customer_RollingAvgOrderValue_12Weeks', 'Customer_RollingUniqueSKUsPurchased_12Weeks', 'Customer_RollingPurchaseCount_12Weeks', 'SKU_RollingTotalSales_12Weeks', 'SKU_RollingPurchaseCount_12Weeks', 'Category_RollingTotalSales_12Weeks', 'Customer_Purchas

In [28]:
skus = pd.read_csv('./data/skus.csv')
customers = pd.read_csv('./data/customers.csv') 
transactions = pd.read_csv('./data/transactions.csv')

## Evaluation

## 📊 Evaluation Metrics Glossary

### 🎯 Recommendation Output & SKU Matching

- **`Top_K`**  
  The number of top recommendations (e.g., top 5 SKUs) evaluated for each customer. It shows how many items we’re suggesting to prioritize.

- **`Top_Predicted_SKUs`**  
  The list of SKUs recommended for a customer, ranked by their predicted purchase likelihood. These are the items we expect the customer to buy.

- **`Historical_Purchases`**  
  The SKUs a customer has previously purchased. This is the customer’s actual buying history for comparison.

- **`Historical_Categories`**  
  The product categories (e.g., Electronics, Clothing) a customer has purchased from. It shows their category preferences.

### 📈 Accuracy & Performance Metrics

- **`SKU_Precision`**  
  The percentage of recommended SKUs that the customer has actually purchased before. Higher precision means our recommendations are accurate.

- **`SKU_Recall`**  
  The percentage of a customer’s historical purchases that appear in our recommendations. Higher recall means we’re capturing more of their past purchases.

- **`SKU_F1_Score`**  
  A balanced measure combining precision and recall. It indicates overall recommendation quality, useful when precision and recall need to be weighed together.

### 🧠 Relevance & Diversity

- **`Category_Overlap_Proportion`**  
  The percentage of recommended SKU categories that match the categories the customer has purchased from. It shows if we’re recommending the right types of products.

- **`Mean_Purchase_Frequency`**  
  The average number of times the recommended SKUs were purchased by the customer in the past. Higher values suggest we’re recommending frequently bought items.

- **`NDCG (Normalized Discounted Cumulative Gain)`**  
  A score (0 to 1) measuring how well we rank relevant SKUs, with higher scores for relevant items appearing at the top of the list. It evaluates the quality of our recommendation order.

- **`MRR (Mean Reciprocal Rank)`**  
  A score (0 to 1) based on the position of the first relevant SKU in the recommendation list. Higher scores mean relevant items are ranked earlier.

### ⏱️ Temporal Insights

- **`Mean_Recency_Weeks`**  
  The average time (in weeks) since the customer last purchased the recommended SKUs. Lower values indicate we’re recommending recently bought items.

- **`Recent_Matches`**  
  The number of recommended SKUs purchased by the customer in the last 2 weeks. It shows if our recommendations align with recent buying behavior.

### 🌐 Category Diversity

- **`Unique_Categories`**  
  The number of different product categories in the recommended SKUs. More categories suggest diverse recommendations.

- **`Category_Entropy`**  
  A measure of how varied the categories are in the recommendations. Higher values indicate more diverse category recommendations, avoiding overly narrow suggestions.

### 📝 Recommendation Context

- **`Validation_Notes`**  
  Detailed explanations for each recommended SKU, noting whether it was purchased before, its category purchase history, or if it’s entirely new. It provides context for why recommendations were made.

- **`Recent_Purchases`**  
  The customer’s most recent purchases (SKUs, categories, and weeks). It helps compare recommendations against what the customer bought lately.

-----------------------

## DEBUGGING

In [ ]:
inference_data[features].describe()